In [1]:

import pandas as pd
from opencc import OpenCC
import numpy as np
import os


inputdirectory = '../../50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/'
outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'

file_Rock = 'Rock_royalties_2021Q1_2025Q2_1_raw_combined.csv' 

lookup_fx = 'lookup_tables/lookup_fx.csv'
lookup_song = 'lookup_tables/lookup_song.csv'
lookup_album = 'lookup_tables/lookup_album.csv'


outputfile = 'Rock_royalties_2021Q1_2025Q2_2_matched.csv'
outputfilexls = 'Rock_royalties_2021Q1_2025Q2_2_matched.xlsx'
converter = OpenCC('s2t') 

def readfile(directory,file):
    path = os.path.join(directory, file)
    df = pd.read_csv(path,low_memory=False)
    print(f"The dataframe of the file '{file}' has {df.shape[0]} rows and {len(df.columns)} columns.")
    return df

def merge(df1, df2, col, what):
    print(f"\nMerging '{what}' on '{col}':")
    empty_cells = df1[col].isna().sum()
    print(f"There are a total of {empty_cells} rows that have no entry in {col}")
    df1.loc[:, col] = df1[col].fillna('XX_UNKNOWN')
    #df1.fillna({col: 'n/a'}, inplace=True)
    df_merged = pd.merge(df1, df2, on=col, how='left')
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    diff_empty = empty_cells2 - empty_cells
    if diff_empty == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {diff_empty} cells that could not be matched (see 'match_issues_{what}_{col}.xlsx').")
        empty_rows = df_merged[df_merged[first_new_col].isna()]
        col=col.replace('/','')
        empty_rows.to_excel(f"{outputdirectory}match_issues_{what}_{col}.xlsx", engine='openpyxl', index=False)
    unused_rows = df2[~df2[col].isin(df_merged[col])]
    unused_rows.to_csv(f'{outputdirectory}unused_lookup_rows_{col}.csv', index=False)  
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 

df_Rock = readfile(inputdirectory, file_Rock)

df_lookup_fx = readfile(inputdirectory, lookup_fx)
df_lookup_song = readfile(inputdirectory ,lookup_song)
df_lookup_album = readfile(inputdirectory, lookup_album)

The dataframe of the file 'Rock_royalties_2021Q1_2025Q2_1_raw_combined.csv' has 859241 rows and 28 columns.
The dataframe of the file 'lookup_tables/lookup_fx.csv' has 161 rows and 13 columns.
The dataframe of the file 'lookup_tables/lookup_song.csv' has 471 rows and 3 columns.
The dataframe of the file 'lookup_tables/lookup_album.csv' has 587 rows and 3 columns.


In [2]:
# pull in the FX rates for the local curreny and compute royalties in HKD and the amount to Rock in HKD
df_Rock_fx = pd.merge(df_Rock, df_lookup_fx[['Report year', 'Report Quarter', 'Currency', 'FX rate']],
                     on=['Report year', 'Report Quarter', 'Currency'],
                     how='left')

df_Rock_fx['SHARE AMOUNT (local FX)'] = pd.to_numeric(df_Rock_fx['SHARE AMOUNT (local FX)'], errors="coerce") # new line 05 Sep 2025
df_Rock_fx['AMOUNT (HKD)']=df_Rock_fx['SHARE AMOUNT (local FX)']/df_Rock_fx['FX rate']
df_Rock_fx['Amount to Rock (HKD)'] = np.where(
    df_Rock_fx['AMOUNT (HKD)'] != 0,  # Condition
    df_Rock_fx['AMOUNT'] * df_Rock_fx['AMOUNT (HKD)'] / df_Rock_fx['SHARE AMOUNT (local FX)'],  # True case
    0  # False case
)

# pull in the FX rates for RMB and compute the amount to Rock in HKD
df_Rock_fx.rename(columns={'FX rate': 'FX rate_main'}, inplace=True)
df_lookup_fx_rmb = df_lookup_fx[df_lookup_fx['Currency'] == 'RMB']
df_Rock_fx = pd.merge(df_Rock_fx, df_lookup_fx_rmb[['Report year', 'Report Quarter', 'FX rate']],
                     on=['Report year', 'Report Quarter'],
                     how='left')
df_Rock_fx.rename(columns={'FX rate': 'FX rate_rmb','FX rate_main': 'FX rate'}, inplace=True)
df_Rock_fx['Amount to Rock (RMB)']=df_Rock_fx['Amount to Rock (HKD)']*df_Rock_fx['FX rate_rmb']

# Checking for empty cells in to be matched columns
empty_cells_p = df_Rock_fx['USER'].isna().sum()    
empty_cells_a = df_Rock_fx['CATALOG TITLE'].isna().sum()
empty_cells_s = df_Rock_fx['SONG TITLE'].isna().sum()
print(f"Empty cells in 'USER', 'CATALOG TITLE', 'SONG TITLE': {empty_cells_p} / {empty_cells_a} / {empty_cells_s}")
df_Rock_fx.fillna({'USER':'XX_UNKNOWN','CATALOG TITLE':'Song Type','SONG TITLE':'XX_UNKNOWN'}, inplace=True)
print(f"The dataframe now has {df_Rock_fx.shape[0]} rows and {len(df_Rock_fx.columns)} columns.")



Empty cells in 'USER', 'CATALOG TITLE', 'SONG TITLE': 0 / 0 / 374
The dataframe now has 859241 rows and 33 columns.


In [3]:
# matching Albums and Songs

df_Rock_fx['SONG TITLE'] = df_Rock_fx['SONG TITLE'].astype(str)
df_Rock_fx['SONG TITLE_mod'] = (df_Rock_fx['SONG TITLE'].str.replace("'", '', regex=False).str.replace(" ", '', regex=False)
            .str.replace("`", '', regex=False).str.replace("，", '', regex=False)
            .str.replace("\t", '', regex=False).str.replace(',', '', regex=False).str.replace(' ', '', regex=False).str.lower())
df_Rock_fx['SONG TITLE_mod'] = df_Rock_fx['SONG TITLE_mod'].apply(converter.convert)

df_Rock_fx['CATALOG TITLE'] = df_Rock_fx['CATALOG TITLE'].astype(str)
df_Rock_fx['CATALOG TITLE_mod'] = df_Rock_fx['CATALOG TITLE'].str.replace("'", '', regex=False).str.replace(',', '', regex=False).str.replace("`", '', regex=False).str.replace(' ', '', regex=False).str.lower()
df_Rock_fx['CATALOG TITLE_mod'] = df_Rock_fx['CATALOG TITLE_mod'].apply(converter.convert)

df_Rock_matched = merge(df_Rock_fx, df_lookup_song, 'SONG TITLE_mod','Rock_fx')
df_Rock_matched = merge(df_Rock_matched, df_lookup_album, 'CATALOG TITLE_mod','df_Rock_matched')

df_Rock_matched=df_Rock_matched.drop(columns=['CATALOG TITLE_mod','SONG TITLE_mod'])

df_Rock_matched.fillna({'Song harmonised':'XX_UNKNOWN','Album (harmonised)':'XX_UNKNOWN','Album type':'XX_UNKNOWN'}, inplace=True)

df_Rock_matched = df_Rock_matched.sort_index(axis=1)
path = os.path.join(outputdirectory, outputfile)
df_Rock_matched.to_csv(path, index=False)
path = os.path.join(outputdirectory, outputfilexls)
df_Rock_matched.to_excel(path, engine='openpyxl', index=True)



Merging 'Rock_fx' on 'SONG TITLE_mod':
There are a total of 0 rows that have no entry in SONG TITLE_mod
Merging of column SONG TITLE_mod was successful
The new dataframe has 859241 rows and 37 columns.

Merging 'df_Rock_matched' on 'CATALOG TITLE_mod':
There are a total of 0 rows that have no entry in CATALOG TITLE_mod
Merging of column CATALOG TITLE_mod was successful
The new dataframe has 859241 rows and 39 columns.


In [4]:
# for col in df_Rock_matched.columns:
#     print(f"Column: {col}")
#     print(df_Rock_matched[col].map(type).value_counts())
#     print()